# 02 -- Damage Detection

Demonstrates the detection adapter: `HeuristicDamageDetector` (always
works, no trained model needed -- analyzes real pixel content: water
color, fire color, edge/crack density) vs. `YOLODamageDetector` (once
your team has trained weights). Both return the same structured shape.

**Important:** the heuristic detector reads image PIXELS, not filenames
-- this notebook proves that by generating synthetic images with generic
names and showing the detector still correctly identifies flood vs. fire
vs. structural content.

In [1]:
import sys
sys.path.append("../backend")

from models.damage_detector import get_detector
import config

print("DEMO_MODE:", config.DEMO_MODE)
detector = get_detector()
print("Detector in use:", type(detector).__name__)

DEMO_MODE: True
Detector in use: HeuristicDamageDetector


## Generate synthetic test images with deliberately generic filenames (no 'flood'/'fire' hints)

In [2]:
import numpy as np
import cv2
import os

os.makedirs("sample_data", exist_ok=True)

# Generic filename, blue-gray water-colored content
img1 = np.zeros((400, 400, 3), dtype=np.uint8)
img1[:, :] = [140, 100, 70]  # BGR blue-gray
cv2.imwrite("sample_data/IMG_0001.jpg", img1)

# Generic filename, orange/red fire-colored content
img2 = np.zeros((400, 400, 3), dtype=np.uint8)
img2[:, :] = [20, 100, 220]  # BGR orange/red
cv2.imwrite("sample_data/IMG_0002.jpg", img2)

# Generic filename, high-frequency noise (simulates rubble/crack texture)
rng = np.random.default_rng(42)
img3 = rng.integers(0, 255, (400, 400, 3), dtype=np.uint8)
cv2.imwrite("sample_data/IMG_0003.jpg", img3)

print("3 synthetic images created with generic filenames")

3 synthetic images created with generic filenames


## Run detection -- notice the results match the actual pixel content, not the filenames

In [3]:
for img_path in ["sample_data/IMG_0001.jpg", "sample_data/IMG_0002.jpg", "sample_data/IMG_0003.jpg"]:
    detections = detector.detect(img_path)
    print(f"\n--- {img_path} ---")
    for d in detections:
        print(f"  {d['damage_type']:<20} object={d['object_type']:<12} damage%={d['damage_percentage']:<4} confidence={d['confidence']}")


--- sample_data/IMG_0001.jpg ---
  flooding             object=road         damage%=100  confidence=0.97

--- sample_data/IMG_0002.jpg ---
  fire                 object=building     damage%=100  confidence=0.97

--- sample_data/IMG_0003.jpg ---
  flooding             object=road         damage%=46   confidence=0.88
  structural_damage    object=building     damage%=100  confidence=0.95


## Video support -- extract frames, detect on each

Videos are broken into up to 5 evenly-spaced frames, each analyzed the
same way as a photo. This is what `/api/upload-batch` does automatically
when you upload a video file.

In [4]:
sys.path.append("../backend")
from services.image_service import extract_frames, cleanup_frames

# Build a short synthetic test video (flood-colored frames)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_path = "sample_data/clip_0001.mp4"
out = cv2.VideoWriter(video_path, fourcc, 10, (400, 400))
for i in range(30):
    frame = np.zeros((400, 400, 3), dtype=np.uint8)
    frame[:, :] = [140, 100, 70]
    out.write(frame)
out.release()

frame_paths = extract_frames(video_path, max_frames=5)
print(f"Extracted {len(frame_paths)} frames from video")

for fp in frame_paths:
    dets = detector.detect(fp)
    print(f"{os.path.basename(fp)}: {[d['damage_type'] for d in dets]}")

cleanup_frames(frame_paths)  # clean up temp frames after use

Extracted 5 frames from video
frame_5.jpg: ['flooding']
frame_10.jpg: ['flooding']
frame_15.jpg: ['flooding']
frame_20.jpg: ['flooding']
frame_25.jpg: ['flooding']


## Using a real trained model

Once your teammates have trained a YOLO model in Colab and exported
weights (e.g. `best.pt`), place the file at
`backend/models/weights/best.pt`, set `DEMO_MODE=false`, and
`get_detector()` will automatically switch to `YOLODamageDetector` --
no other code changes needed.

In [5]:
# Example (will raise if weights don't exist yet -- that's expected until trained):
# from models.damage_detector import YOLODamageDetector
# real_detector = YOLODamageDetector("../backend/models/weights/best.pt")
# real_detector.detect("path/to/real_image.jpg")